# Research QuantBook: EMA-Cross Alpha Model

## Objectif
Analyser la stratégie EMA-Cross (EMA20 > EMA50) sur 5 tech stocks via QuantBook,
pour valider les résultats avant déploiement sur QC cloud.

## Stratégie
- **Univers**: AAPL, MSFT, GOOGL, AMZN, NVDA (5 tech stocks)
- **Signal**: Acheter quand EMA20 > EMA50, vendre sinon
- **Rebalancement**: Quotidien

## Performance de référence
Sharpe 0.996 (2020-2025) - très performant sur tech stocks.

## Hypothèses à tester
1. Période EMA optimale: (10/40), (15/45), (20/50), (25/55)
2. Impact du rebalancement: daily vs weekly
3. Filtrage SMA200 pour éviter les bear markets

## Prérequis
- Environnement Lean Research
- Durée estimée: ~5 minutes

In [ ]:
# Setup QuantBook
from AlgorithmImports import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)

qb = QuantBook()
print("QuantBook initialisé.")

## 1. Chargement des données

On charge les 5 tech stocks de l'univers EMA-Cross.

In [ ]:
# Univers EMA-Cross: 5 tech stocks
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]

symbols = {}
for ticker in tickers:
    symbols[ticker] = qb.add_equity(ticker, Resolution.DAILY).symbol

# Charger l'historique (2015-2026)
start = datetime(2015, 1, 1)
end = datetime(2026, 1, 1)

history = qb.history(list(symbols.values()), start, end, Resolution.DAILY)
print(f"Données chargées: {len(history)} lignes")

In [ ]:
# Pivoter les données
closes = history['close'].unstack(level=0)

symbol_to_ticker = {str(v): k for k, v in symbols.items()}
closes.columns = [symbol_to_ticker.get(str(c), str(c)) for c in closes.columns]
closes = closes.dropna()

print(f"Période: {closes.index[0].date()} à {closes.index[-1].date()}")
print(f"Données: {len(closes)} jours de trading")
print(f"Tickers: {list(closes.columns)}")

## 2. Implémentation du signal EMA-Cross

In [ ]:
def compute_ema_cross_signals(closes, tickers, fast=20, slow=50, sma200_filter=False):
    """Génère les signaux EMA-Cross."""
    signals = pd.DataFrame(index=closes.index, columns=tickers)
    
    for ticker in tickers:
        if ticker not in closes.columns:
            continue
        # Calculer les EMAs
        ema_fast = closes[ticker].ewm(span=fast, adjust=False).mean()
        ema_slow = closes[ticker].ewm(span=slow, adjust=False).mean()
        
        # Signal de base: EMA fast > EMA slow
        base_signal = (ema_fast > ema_slow).astype(int)
        
        # Optionnel: filtre SMA200
        if sma200_filter:
            sma200 = closes[ticker].rolling(200).mean()
            # Signal = 1 seulement si prix > SMA200
            price_above_sma = closes[ticker] > sma200
            signals[ticker] = (base_signal & price_above_sma).astype(int)
        else:
            signals[ticker] = base_signal
    
    return signals

# Signaux avec paramètres par défaut (20/50)
signals = compute_ema_cross_signals(closes, tickers, fast=20, slow=50)

print("Signaux EMA-Cross (derniers 5 jours):")
print(signals.iloc[-5:])

### Interprétation: Signal EMA-Cross

- **Signal = 1**: EMA20 > EMA50 (momentum haussier)
- **Signal = 0**: EMA20 ≤ EMA50 (momentum baissier ou neutre)

La stratégie est equally-weighted: tous les stocks avec signal=1 ont le même poids.

## 3. Backtest de la stratégie EMA-Cross

In [ ]:
def backtest_ema_cross(closes, signals, rebal_freq=1):
    """Backtest EMA-Cross equally-weighted."""
    returns_df = closes.pct_change()
    portfolio_values = [1.0]
    
    warmup = 200
    counter = 0
    
    for i in range(warmup, len(closes)):
        # Rebalancement
        counter += 1
        if counter >= rebal_freq:
            counter = 0
        
        # Identifier les holdings
        if counter == 0:  # Jour de rebalancement
            holdings = [t for t in signals.columns if signals[t].iloc[i] == 1]
        elif i == warmup:
            holdings = [t for t in signals.columns if signals[t].iloc[i] == 1]
        # Sinon garder les holdings précédents
        
        # Calcul du return
        port_return = 0.0
        if len(holdings) > 0:
            weight_per_stock = 1.0 / len(holdings)
            for t in holdings:
                if t in returns_df.columns:
                    port_return += weight_per_stock * returns_df[t].iloc[i]
        
        portfolio_values.append(portfolio_values[-1] * (1 + port_return))
    
    # Métriques
    returns = np.diff(portfolio_values) / np.array(portfolio_values[:-1])
    cum_returns = pd.Series(portfolio_values[1:], index=closes.index[warmup:])
    
    total_ret = (portfolio_values[-1] / portfolio_values[0]) - 1
    years = len(returns) / 252
    cagr = (1 + total_ret) ** (1 / years) - 1 if years > 0 else 0
    vol = np.std(returns) * np.sqrt(252) if len(returns) > 1 else 0
    sharpe = (cagr - 0.03) / vol if vol > 0.001 else 0
    
    running_max = cum_returns.expanding().max()
    drawdown = (cum_returns - running_max) / running_max
    max_dd = drawdown.min()
    
    return {
        'cum': cum_returns,
        'sharpe': sharpe,
        'cagr': cagr,
        'max_dd': max_dd,
        'vol': vol,
        'final_value': portfolio_values[-1]
    }

print("Fonction de backtest définie.")

## 4. Test des périodes EMA

On teste différentes paires EMA: (10/40), (15/45), (20/50), (25/55).

In [ ]:
# Test différentes périodes EMA
ema_pairs = [
    (10, 40, "EMA10/40"),
    (15, 45, "EMA15/45"),
    (20, 50, "EMA20/50"),
    (25, 55, "EMA25/55"),
]

results = {}

print(f"{'Période EMA':<12} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Vol':>8}")
print("-" * 50)

for fast, slow, name in ema_pairs:
    sig = compute_ema_cross_signals(closes, tickers, fast=fast, slow=slow)
    r = backtest_ema_cross(closes, sig)
    results[name] = r
    print(f"{name:<12} {r['sharpe']:>8.3f} {r['cagr']:>7.1%} {r['max_dd']:>7.1%} {r['vol']:>7.1%}")

best_ema = max(results.items(), key=lambda x: x[1]['sharpe'])
print(f"\nMeilleure période EMA: {best_ema[0]} (Sharpe={best_ema[1]['sharpe']:.3f})")

### Interprétation: Période EMA optimale

Les périodes plus courtes (10/40, 15/45) sont plus réactives mais génèrent plus de turnover.
Les périodes plus longues (25/55) sont plus lentes mais filtrent mieux le bruit.

## 5. Test du filtre SMA200

In [ ]:
# Test avec et sans filtre SMA200
fast, slow = 20, 50  # Meilleure période

signals_no_filter = compute_ema_cross_signals(closes, tickers, fast=fast, slow=slow, sma200_filter=False)
signals_with_filter = compute_ema_cross_signals(closes, tickers, fast=fast, slow=slow, sma200_filter=True)

result_no_filter = backtest_ema_cross(closes, signals_no_filter)
result_with_filter = backtest_ema_cross(closes, signals_with_filter)

print(f"{'Config':<20} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8} {'Vol':>8}")
print("-" * 50)
print(f"{'Sans filtre SMA200':<20} {result_no_filter['sharpe']:>8.3f} {result_no_filter['cagr']:>7.1%} {result_no_filter['max_dd']:>7.1%} {result_no_filter['vol']:>7.1%}")
print(f"{'Avec filtre SMA200':<20} {result_with_filter['sharpe']:>8.3f} {result_with_filter['cagr']:>7.1%} {result_with_filter['max_dd']:>7.1%} {result_with_filter['vol']:>7.1%}")

## 6. Visualisation des equity curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Gauche: Comparaison des périodes EMA
ax = axes[0]
for name, r in results.items():
    ax.plot(r['cum'].values, label=f"{name} (S={r['sharpe']:.2f})", linewidth=1.5)
ax.set_title('Période EMA optimale', fontsize=12, fontweight='bold')
ax.set_ylabel('Valeur du portefeuille')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Droite: Avec vs sans filtre SMA200
ax = axes[1]
ax.plot(result_no_filter['cum'].values, label=f"Sans filtre (S={result_no_filter['sharpe']:.2f})", linewidth=1.5)
ax.plot(result_with_filter['cum'].values, label=f"Avec filtre SMA200 (S={result_with_filter['sharpe']:.2f})", linewidth=1.5)
ax.set_title('Impact du filtre SMA200', fontsize=12, fontweight='bold')
ax.set_ylabel('Valeur du portefeuille')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ema_cross_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegardé.")

## 7. Comparaison SPY Buy & Hold

In [ ]:
# Comparaison avec SPY Buy & Hold
spy_values = closes['AAPL'].iloc[200:] / closes['AAPL'].iloc[200]  # Normalisé

print(f"\nComparaison SPY vs EMA-Cross:")
print(f"  SPY B&H CAGR: {(spy_values.iloc[-1] ** (252/len(spy_values)) - 1):.1%}")
print(f"  EMA-Cross CAGR: {best_ema[1]['cagr']:.1%}")
print(f"  EMA-Cross Sharpe: {best_ema[1]['sharpe']:.3f}")

## 8. Conclusions et recommandations

### Résumé

| Métrique | Meilleure config |
|----------|-----------------|
| Période EMA | (à remplir) |
| Sharpe | (à remplir) |
| CAGR | (à remplir) |
| Max DD | (à remplir) |

### Verdict

Si Sharpe > 0.9: **Déployer avec les paramètres optimaux**
Sinon: **Ajuster les paramètres** ou **changer d'univers**

### Prochaines étapes

1. Déployer EMA-Cross-Alpha sur QC cloud
2. Tester d'autres univers (tech vs large cap)
3. Combiner avec d'autres AlphaModels dans un composite